# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YashikaChandra06/flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def classify_url(url):
    reasons = []

    suspicious_words = ["login", "verify", "free", "prize", "urgent"]

    if any(word in url.lower() for word in suspicious_words):
        reasons.append("RC01: Suspicious keyword")

    if len(url) > 50:
        reasons.append("RC02: Long URL")

    special_chars = sum(not c.isalnum() for c in url)
    if special_chars > 8:
        reasons.append("RC03: Too many special characters")

    if len(reasons) >= 2:
        prediction = "Scam"
        reasons.append("RC04: Multiple suspicious indicators")
    else:
        prediction = "Legitimate"
        if not reasons:
            reasons.append("RC05: No suspicious indicators")

    return prediction, reasons


prediction, reason_codes = classify_url(
    "https://free-prize-login12345.com/verify/account"
)

print("Prediction:", prediction)
print("Reason Codes:")
for code in reason_codes:
    print("-", code)

Prediction: Legitimate
Reason Codes:
- RC01: Suspicious keyword


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd

# Create a copy
baseline = df.copy() # type: ignore

# ----- Baseline scoring rule -----
# Example: Higher priority for pages with higher clicks and lower CTR.
# Replace these columns with the actual ones in your dataset if needed.

baseline["baseline_action_score"] = (
    baseline["clicks"] * (1 - baseline["ctr"])
)

# Rank pages (highest score = highest priority)
baseline = baseline.sort_values(
    by="baseline_action_score",
    ascending=False
).reset_index(drop=True)

baseline["rank"] = baseline.index + 1

# Select output columns
output = baseline[
    ["rank", "baseline_action_score"] +
    [col for col in baseline.columns
     if col not in ["rank", "baseline_action_score"]]
]

# Create output folder
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Save CSV
output_path = output_dir / "baseline_action_score.csv"
output.to_csv(output_path, index=False)

print(f"Saved ranked queue to: {output_path}")
output.head(10)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Load the ranked queue
review = pd.read_csv("work/outputs/baseline_action_score.csv")

# Keep the top 20 rows
top20 = review.head(20).copy()

# Action
top20["action"] = "Refresh Content"

# Reason code
top20["reason_code"] = "RC01"

# Confidence note
top20["confidence_note"] = (
    "Medium - Based on the baseline action score only."
)

# What would make it wrong?
top20["what_would_make_it_wrong"] = (
    "Recent content update, missing signals, seasonal traffic, "
    "or important features not included in the baseline rule."
)

# Display review table
top20[[
    "rank",
    "baseline_action_score",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]]

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Inspect available columns
# ensure df is defined (load from data_path if needed) and print columns
if 'df' not in globals():
    try:
        df = pd.read_csv(data_path)
    except FileNotFoundError:
        # fallback: try the canonical relative path or fail with informative message
        alt = Path("data/raw/content_refresh_anonymized.csv")
        if alt.exists():
            df = pd.read_csv(alt)
        else:
            # This cell is for CODE (numbers, a query, a check).
            # Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

            # Inspect available columns
            # ensure df is defined (load from data_path if needed) and print columns
            if 'df' not in globals():
                try:
                    df = pd.read_csv(data_path)
                except FileNotFoundError:
                    # fallback: try the canonical relative path or fail with informative message
                    alt = Path("data/raw/content_refresh_anonymized.csv")
                    if alt.exists():
                        df = pd.read_csv(alt)
                    else:
                        # try searching upward for the file in parent directories before failing
                        found = None
                        for parent in [Path.cwd()] + list(Path.cwd().parents):
                            candidate = parent / data_path
                            if candidate.exists():
                                found = candidate
                                break
                            candidate2 = parent / alt
                            if candidate2.exists():
                                found = candidate2
                                break

                        if found:
                            df = pd.read_csv(found)
                        else:
                            raise FileNotFoundError(
                                f"Could not find dataset at {data_path} or fallback {alt}. "
                                f"Current working dir: {Path.cwd()}. Searched parent directories."
                            )
            print(df.columns.tolist())

            # Look for possible leakage-related columns
            leakage_keywords = [
                "future", "after", "post", "next",
                "outcome", "label", "target", "converted",
                "product", "manual", "flag"
            ]

            possible_leakage = [
                c for c in df.columns
                if any(k in c.lower() for k in leakage_keywords)
            ]

            print("Possible leakage columns:", possible_leakage)
print(df.columns.tolist())

# Look for possible leakage-related columns
leakage_keywords = [
    "future", "after", "post", "next",
    "outcome", "label", "target", "converted",
    "product", "manual", "flag"
]

possible_leakage = [
    c for c in df.columns
    if any(k in c.lower() for k in leakage_keywords)
]

print("Possible leakage columns:", possible_leakage)

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Possible leakage columns: []
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'cl

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.